<a href="https://colab.research.google.com/github/akinns247/Starter_Notebook247/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akinns247/Starter_Notebook247/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
## Baseline Rule: Content Refresh Prioritization

My baseline rule identifies pages that may need content refresh by using historical performance signals.

The rule uses:
- CTR opportunity: Pages ranking well but receiving fewer clicks than expected may need title, meta description, or content improvements.
- Search demand: Pages with higher search volume have more potential value if improved.

Reason codes:
- CTR_OPPORTUNITY: Page has strong ranking position but lower click-through rate.
- HIGH_VALUE_REFRESH: Page has higher search demand and improvement opportunity.
- MONITOR: Page does not strongly match refresh signals.

The score is a baseline decision-support ranking. It helps a content manager prioritize pages for review, but it does not automatically decide which pages should be updated.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import os

# TODO: Load your actual dataset into a pandas DataFrame named 'df'.
# For example: df = pd.read_csv('your_data.csv')
# For demonstration, creating a sample DataFrame with required columns.
df = pd.DataFrame({
    'content_id': [f'page_{i}' for i in range(1, 11)],
    'ctr': [0.01, 0.05, 0.03, 0.08, 0.02, 0.06, 0.04, 0.09, 0.07, 0.015],
    'avg_position': [12, 5, 8, 3, 15, 7, 10, 2, 6, 11],
    'search_volume': [1000, 5000, 2000, 8000, 1500, 6000, 3000, 9000, 4000, 1200]
}) # Replaced placeholder with sample data

# Create a copy of the dataset
baseline = df.copy()

# Create baseline score
baseline["baseline_score"] = 0

# Signal 1: CTR opportunity
baseline["baseline_score"] += (
    (baseline["ctr"] < baseline["ctr"].median()) &
    (baseline["avg_position"] <= 10)
).astype(int) * 50

# Signal 2: Search demand opportunity
baseline["baseline_score"] += (
    baseline["search_volume"] > baseline["search_volume"].median()
).astype(int) * 30


# Assign reason codes
def assign_reason(row):
    reasons = []

    if row["ctr"] < baseline["ctr"].median() and row["avg_position"] <= 10:
        reasons.append("CTR_OPPORTUNITY")

    if row["search_volume"] > baseline["search_volume"].median():
        reasons.append("HIGH_VALUE_REFRESH")

    if len(reasons) == 0:
        return "MONITOR"

    return "_".join(reasons)


baseline["reason_code"] = baseline.apply(assign_reason, axis=1)


# Action label
baseline["action"] = baseline["baseline_score"].apply(
    lambda x: "REFRESH_REVIEW" if x >= 50 else "MONITOR"
)


# Rank pages
baseline = baseline.sort_values(
    "baseline_score",
    ascending=False
)

baseline["rank"] = range(1, len(baseline)+1)


# Save CSV
os.makedirs("work/outputs", exist_ok=True)

baseline[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)


# Display top 20
baseline.head(20)

,content_id,ctr,avg_position,search_volume,baseline_score,reason_code,action,rank
6,page_7,0.040,10,3000,50,CTR_OPPORTUNITY,REFRESH_REVIEW,1
2,page_3,0.030,8,2000,50,CTR_OPPORTUNITY,REFRESH_REVIEW,2
8,page_9,0.070,6,4000,30,HIGH_VALUE_REFRESH,MONITOR,3
1,page_2,0.050,5,5000,30,HIGH_VALUE_REFRESH,MONITOR,4
5,page_6,0.060,7,6000,30,HIGH_VALUE_REFRESH,MONITOR,5
3,page_4,0.080,3,8000,30,HIGH_VALUE_REFRESH,MONITOR,6
7,page_8,0.090,2,9000,30,HIGH_VALUE_REFRESH,MONITOR,7
0,page_1,0.010,12,1000,0,MONITOR,MONITOR,8
4,page_5,0.020,15,1500,0,MONITOR,MONITOR,9
9,page_10,0.015,11,1200,0,MONITOR,MONITOR,10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
## Top-20 Review

The top-ranked pages were selected because they showed stronger refresh signals based on the baseline rule.

For each page:
- Action: REFRESH_REVIEW means the page should be reviewed by a content manager.
- Reason code explains the signal behind the recommendation.
- Confidence note explains why the page was selected.
- What would make it wrong explains possible limitations.

The recommendations are directional and require human review before action.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
## Weak Picks and Leakage Check

Some recommendations may be weak because the baseline uses only a small number of historical signals.

Weak picks may include:
- Pages with low CTR caused by different search intent rather than content quality.
- Pages with high search volume but limited improvement opportunity.

Leakage check:
- No future performance data was used.
- No model labels were used.
- No product flags were used.
- Only historical page performance signals were used.

This baseline provides a simple comparison point for the Week 5 machine learning model.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.